# Evaluation

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from wikifin_rag.rag_helper import RAGBase
from wikifin_rag.embedder import Embedder
from wikifin_rag.evaluation_utils import generate_corpus_ground_truth, evaluate
from wikifin_rag.config import PROJECT_ROOT
import pandas as pd
from psycopg import sql
import os

## Offline Evaluation Datasets
### Retrieval evaluation data

In [2]:
embedder = Embedder()

load_dotenv(override=True)
openai_client = OpenAI()

assistant = RAGBase(embedder=embedder, llm_client=openai_client)
db_client = assistant.db_client

In [3]:
def retrieve_documents(n=100):
    db_client.open_connection()
    
    db_client.cur.execute(
        sql.SQL(
            """
            SELECT
                c.document_id || '_' || c.chunk_id AS id,
                d.title,
                d.section,
                c.content
            FROM chunks c
            JOIN documents d
            ON c.document_id = d.id
            WHERE language = 'nl'
            LIMIT %s;
            """
        ),
        (n,)
    )
    documents = db_client.cur.fetchall()

    db_client.close_connection()

    return documents

In [4]:
# sets the location where the ground truth dataset is stored
dest = PROJECT_ROOT / "data"
filename = "ground_truth.csv"

# determines the size of the dataset
n_docs = 300 # number of documents to run the evaluation on
n_q = 5 # number of questions to generate per document

In [5]:
# TODO: set to True to rerun the data generation function
refresh_data = False

In [6]:
documents = retrieve_documents(n=n_docs)

In [7]:
if refresh_data or not os.path.exists(dest / filename):
    _, total_cost = generate_corpus_ground_truth(documents, llm_client=openai_client, n=n_q)

In [8]:
df_ground_truth = pd.read_csv(dest / filename)
ground_truth = df_ground_truth.to_dict(orient="records")

### RAG evaluation data

In [ ]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["id"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [15]:
ground_truth[0]

{'question': 'Welke kosten en taksen moet je betalen als je in een tak 23-levensverzekering belegt?',
 'document': '0a3394ca9c1625e8_0'}

In [ ]:
# answer_record = generate_rag_answer(ground_truth[0])

KeyError: 'answer'

## Retrieval metrics
### Text Search

In [ ]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [12]:
from wikifin_rag.evaluation_utils import compute_relevance, compute_relevance_total

In [18]:
db_client.open_connection()

In [19]:
def text_search(query):
    return assistant.db_client.text_search(query=query, num_results=5)

In [ ]:
def objective():
    retrieval_metrics = evaluate(ground_truth, text_search)
    return {'loss': -retrieval_metrics['mrr'], 'status': STATUS_OK }

In [ ]:
search_space = {
    "A": hp.loguniform("A", -5, 0),
    "B": hp.loguniform("B", -7, 0),
    "C": hp.loguniform("C", -3, 0),
    "D": 1 - hp.uniform("D_raw", 0, 1) ** 2
}

### Vector Search

In [ ]:
def vector_search(query):
    return assistant.db_client.vector_search(query=query, num_results=5)

### Hybrid Search

In [ ]:
def hybrid_search(query):
    return assistant.search(query=query, num_results=5)